In [0]:
# Gold Layer - Business KPIs and aggregations
from pyspark.sql.functions import *

# ── Read Silver Table ─────────────────────────────────────
silver_df = spark.read.format("delta").table("retail_shelf_intelligence.raw_data.silver_retail")

# ── Gold Table 1: Stock Risk by City and Category ─────────
stock_risk = silver_df \
    .groupBy("City", "Category") \
    .agg(
        count("Invoice_ID").alias("Total_Transactions"),
        avg("Stock_On_Hand").alias("Avg_Stock"),
        avg("Reorder_Level").alias("Avg_Reorder_Level"),
        sum("Stock_Risk_Flag").alias("Risk_Count"),
        avg("Units").alias("Avg_Units_Sold"),
        sum("Revenue").alias("Total_Revenue")
    ) \
    .withColumn("Stock_Risk_%", 
        round((col("Risk_Count") / col("Total_Transactions")) * 100, 2))

stock_risk.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_shelf_intelligence.raw_data.gold_stock_risk")

print(f"Gold Stock Risk saved: {stock_risk.count()} rows")

# ── Gold Table 2: Sales by Weather Condition ──────────────
weather_sales = silver_df \
    .groupBy("City", "Category") \
    .agg(
        avg("temperature_2m_max").alias("Avg_Max_Temp"),
        avg("precipitation_sum").alias("Avg_Precipitation"),
        sum("Revenue").alias("Total_Revenue"),
        sum("Units").alias("Total_Units_Sold")
    )

weather_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_shelf_intelligence.raw_data.gold_weather_sales")

print(f"Gold Weather Sales saved: {weather_sales.count()} rows")

# ── Gold Table 3: Holiday Impact ──────────────────────────
holiday_impact = silver_df \
    .groupBy("is_holiday", "Category") \
    .agg(
        sum("Revenue").alias("Total_Revenue"),
        sum("Units").alias("Total_Units"),
        count("Invoice_ID").alias("Total_Transactions"),
        avg("Stock_On_Hand").alias("Avg_Stock")
    )

holiday_impact.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_shelf_intelligence.raw_data.gold_holiday_impact")

print(f"Gold Holiday Impact saved: {holiday_impact.count()} rows")

# ── Gold Table 4: Brand Performance ──────────────────────
brand_performance = silver_df \
    .groupBy("Brand", "Category", "City") \
    .agg(
        sum("Revenue").alias("Total_Revenue"),
        sum("Units").alias("Total_Units"),
        avg("Margin_%").alias("Avg_Margin_%"),
        avg("Stock_On_Hand").alias("Avg_Stock"),
        sum("Stock_Risk_Flag").alias("Stock_Risk_Count")
    )

brand_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retail_shelf_intelligence.raw_data.gold_brand_performance")

print(f"Gold Brand Performance saved: {brand_performance.count()} rows")

print("\n✅ Gold layer complete!")